In [82]:
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle


In [22]:
orig_df = pd.read_csv('../data/csv/master_df.csv')
master_df = orig_df.copy()
master_df.head(1)

,id,href,price,title,status,content,createdAt,boostedAt,user_dbId,user_nickname,...,boost_diff_seconds,is_boosted,title_len,content_len,has_keyword_new,has_keyword_urgent,price_ratio_to_brand,price_ratio_to_label,days_elapsed,target_n_days
0,11121x6p7597,https://www.daangn.com/kr/buy-sell/%EB%A7%88%E...,22000,마리마켓 트렌치자켓 롤업팬츠 세트 S,Ongoing,"마리마켓 트렌치자켓 롤업팬츠 세트\n베이지 색상\n자켓, 팬츠 둘다 사이즈 S\n미...",2025-10-13 11:00:57+00:00,2026-03-03 12:02:21+00:00,24156105,리본,...,12186084.0,1,20,61,0,0,0.223793,0.122495,142.831007,0.0


In [23]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220688 entries, 0 to 220687
Data columns (total 57 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        220688 non-null  str    
 1   href                      220688 non-null  str    
 2   price                     220688 non-null  int64  
 3   title                     220688 non-null  str    
 4   status                    220688 non-null  str    
 5   content                   220688 non-null  str    
 6   createdAt                 220688 non-null  str    
 7   boostedAt                 220688 non-null  str    
 8   user_dbId                 220688 non-null  int64  
 9   user_nickname             220688 non-null  str    
 10  region_name_from_article  220688 non-null  str    
 11  region_id                 220688 non-null  int64  
 12  region_name               220688 non-null  str    
 13  region_in                 220688 non-null  str    
 14 

In [24]:
master_df['status'].value_counts()

status
Ongoing     146835
Closed       70958
Reserved      2895
Name: count, dtype: int64

In [ ]:
cols_to_drop = [
    'boost_diff_seconds',
    'img_path',
    'review_reason',
    'triage',
    'top3_scores',
    'top3_labels',
    'fine_ratio',
    'fine_margin',
    'fine_conf',
    'coarse_ratio',
    'coarse_margin',
    'coarse_conf',
    'label_conf',
    'fashion_keywords',
    'filled_count',
    'source_file',
    'error',
    'status_detail',
    'imagePath',
    'imageFile',
    'imageUrls',
    'region_in',
    'region_id',
    'region_name_from_article',
]

In [ ]:
clean_df = master_df.drop(cols_to_drop, axis=1)
clean_df.to_csv('../data/csv/clean_df.csv', index=False)
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220688 entries, 0 to 220687
Data columns (total 33 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    220688 non-null  str    
 1   href                  220688 non-null  str    
 2   price                 220688 non-null  int64  
 3   title                 220688 non-null  str    
 4   status                220688 non-null  str    
 5   content               220688 non-null  str    
 6   createdAt             220688 non-null  str    
 7   boostedAt             220688 non-null  str    
 8   user_dbId             220688 non-null  int64  
 9   user_nickname         220688 non-null  str    
 10  region_name           220688 non-null  str    
 11  crawledAt             220688 non-null  str    
 12  favoriteCount         220688 non-null  int64  
 13  chatCount             220688 non-null  int64  
 14  viewCount             220688 non-null  int64  
 15  sellerTempe

In [63]:
n_days = 7  # 예: 7일 이내 판매 여부


# 0. datetime 타입으로 변환 (timezone 제거 혹은 통일)
clean_df['createdAt'] = pd.to_datetime(clean_df['createdAt'], utc=True, format='mixed')
clean_df['boostedAt'] = pd.to_datetime(clean_df['boostedAt'], utc=True, format='mixed')
clean_df['crawledAt'] = pd.to_datetime(clean_df['crawledAt'], utc=True, format='mixed')


# 1. 경과 시간(일) 계산
clean_df['days_elapsed'] = (
    clean_df['crawledAt'] - clean_df['createdAt']
).dt.total_seconds() / (24 * 3600)


# 2. Target 변수 생성 함수
def make_target(row, n):
    # status 컬럼의 '거래완료'를 의미하는 정확한 텍스트 확인 필요 (예: 'Completed', 'Sold' 등)
    is_sold = row['status'] != 'Ongoing'  # 혹은 status_detail 사용

    if is_sold and row['days_elapsed'] <= n:
        return 1  # n일 이내 판매됨
    elif not is_sold and row['days_elapsed'] > n:
        return 0  # n일이 지났는데도 안 팔림
    else:
        return np.nan  # 아직 n일이 안 지났는데 안 팔린 상태 (보류)


clean_df['target_n_days'] = clean_df.apply(lambda x: make_target(x, n_days), axis=1)

# 판별 불가능한(보류된) 데이터 학습에서 제외
train_df = clean_df.dropna(subset=['target_n_days']).copy()
train_df['target_n_days'] = train_df['target_n_days'].astype(int)


In [64]:
# 1. 비율 관련 피처 (조회수 0으로 나누는 것 방지)
train_df['favorite_per_view'] = train_df['favoriteCount'] / (train_df['viewCount'] + 1)
train_df['chat_per_view'] = train_df['chatCount'] / (train_df['viewCount'] + 1)

# 2. 가격 로그 변환 (이상치 영향을 줄이기 위함)
train_df['price_log'] = np.log1p(train_df['price'])

In [ ]:
cols = [
    # --- 판매자 관점에서 볼 수 있는 현재 반응도 데이터 ---
    'status',
    'price',
    'price_log',
    'price_ratio_to_brand',
    'sellerTemperature',
    'title_len',
    'has_keyword_new',
    'title',
    'content',
    'region_name',
    # --- 구매자 관점에서 볼 수 있는 현재 반응도 데이터 ---
    'viewCount',
    'favoriteCount',
    'chatCount',
    'favorite_per_view',
    'chat_per_view',
    'is_boosted',
    'target_n_days',
]

In [66]:
train_df = train_df[cols].copy()

In [67]:
train_df.to_csv('../data/csv/train_df.csv', index=False)

In [ ]:
df = pd.read_csv('../data/csv/train_df2.csv')
df.head(1)

,status,price,price_log,price_ratio_to_brand,sellerTemperature,title_len,has_keyword_new,title,content,region_name,viewCount,favoriteCount,chatCount,favorite_per_view,chat_per_view,is_boosted,target_n_days
0,Ongoing,22000,9.998843,0.223793,75.9,20,0,마리마켓 트렌치자켓 롤업팬츠 세트 S,"마리마켓 트렌치자켓 롤업팬츠 세트\n베이지 색상\n자켓, 팬츠 둘다 사이즈 S\n미...",역삼1동,168,3,0,0.017751,0.0,1,0


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# 한국어 문장 임베딩에 성능이 좋은 모델 로드 (GPU가 있다면 자동으로 할당됩니다)
# 처음 실행 시 모델을 다운로드하므로 시간이 조금 걸릴 수 있습니다.
model_name = 'jhgan/ko-sroberta-multitask'
embedder = SentenceTransformer(model_name)


def create_text_embeddings(df, text_columns):
    """
    지정된 텍스트 컬럼들을 임베딩 벡터로 변환하여 기존 데이터프레임에 병합합니다.
    """
    df_embedded = df.copy()

    for col in text_columns:
        print(f'[{col}] 임베딩 추출 중...')
        # 텍스트 결측치 처리 (빈 문자열로)
        texts = df_embedded[col].fillna('').tolist()

        # 임베딩 생성 (768차원 벡터로 변환됨)
        embeddings = embedder.encode(texts, show_progress_bar=True)

        # 임베딩 결과를 데이터프레임 컬럼으로 변환
        emb_df = pd.DataFrame(
            embeddings,
            columns=[f'{col}_emb_{i}' for i in range(embeddings.shape[1])],
            index=df_embedded.index,
        )

        # 기존 텍스트 컬럼 삭제 및 임베딩 컬럼 병합
        df_embedded = df_embedded.drop(columns=[col])
        df_embedded = pd.concat([df_embedded, emb_df], axis=1)

    return df_embedded


# 사용 예시 (df는 원본 데이터프레임)
text_cols = ['title', 'content']
#df_vectorized = create_text_embeddings(df, text_cols)

#df_vectorized.to_csv('./data/csv/vectorized_df.csv', index=False)

In [74]:
vec_df = pd.read_csv('../data/csv/vectorized_df.csv')

vec_df.head(1)

,status,price,price_log,price_ratio_to_brand,sellerTemperature,title_len,has_keyword_new,region_name,viewCount,favoriteCount,...,content_emb_758,content_emb_759,content_emb_760,content_emb_761,content_emb_762,content_emb_763,content_emb_764,content_emb_765,content_emb_766,content_emb_767
0,Ongoing,22000,9.998843,0.223793,75.9,20,0,역삼1동,168,3,...,-0.096123,-0.015974,-0.284907,0.890095,0.412497,-0.284411,-0.07036,0.08028,-0.071742,-0.379537


In [75]:
vec_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 131842 entries, 0 to 131841
Columns: 1551 entries, status to content_emb_767
dtypes: float64(1541), int64(8), str(2)
memory usage: 1.5 GB


In [ ]:
cols_to_drop = [
    # --- 판매자 관점에서 볼 수 있는 현재 반응도 데이터 ---
    'status',
    'price',
    # --- 구매자 관점에서 볼 수 있는 현재 반응도 데이터 ---
    'viewCount',
    'favoriteCount',
    'chatCount',
    'favorite_per_view',
    'chat_per_view',
    'is_boosted',
    'target_n_days',
]

In [84]:
X_train = vec_df.drop(cols_to_drop, axis=1)
y_train = vec_df['target_n_days']

categorical_features = ['region_name']

In [ ]:
import numpy as np

# (주의: X_train, y_train, categorical_features는 사전에 정의되어 있다고 가정합니다)


# 1. Optuna용 앙상블 학습 함수 (인자로 cat_features 추가)
def train_ensemble_for_optuna(X_tr, y_tr, params, cat_features, n_models=5):
    majority_idx = y_tr[y_tr == 0.0].index
    minority_idx = y_tr[y_tr == 1.0].index

    majority_idx_shuffled = shuffle(majority_idx, random_state=42)
    majority_chunks = np.array_split(majority_idx_shuffled, n_models)

    models = []
    for chunk_idx in majority_chunks:
        train_idx = chunk_idx.tolist() + minority_idx.tolist()

        X_subset = X_tr.loc[train_idx]
        y_subset = y_tr.loc[train_idx]

        # 범주형 데이터 지정
        train_pool = Pool(data=X_subset, label=y_subset, cat_features=cat_features)

        model = CatBoostClassifier(**params)
        model.fit(train_pool, verbose=False)
        models.append(model)

    return models


# 2. 앙상블 예측 함수 (🌟 수정됨: 에러 방지를 위해 Pool 객체 사용)
def predict_ensemble(models, X_val, cat_features):
    predictions = np.zeros(len(X_val))
    # 예측할 때도 데이터프레임이 아닌 Pool 형태로 넣어줘야 CatBoost가 범주형을 인식합니다.
    test_pool = Pool(data=X_val, cat_features=cat_features)

    for model in models:
        predictions += model.predict_proba(test_pool)[:, 1]
    return predictions / len(models)


# 3. 데이터 분할 (🌟 수정됨: Optuna 평가의 일관성을 위해 밖에서 한 번만 분할)
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)


# 4. Optuna 목적 함수(Objective Function) 정의
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 15.0),
        'random_strength': trial.suggest_float('random_strength', 0.01, 1.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'loss_function': 'Logloss',
        'eval_metric': 'Logloss',
        'task_type': 'GPU',
        'bootstrap_type': 'Bayesian',  # 🌟 수정됨: GPU 환경에서 bagging_temperature 쓸 때 필수
        'verbose': False,
    }

    # 미리 고정해둔 분할 데이터를 넣어서 학습
    models = train_ensemble_for_optuna(
        X_opt_tr, y_opt_tr, params, categorical_features, n_models=5
    )

    # 예측 수행
    val_proba = predict_ensemble(models, X_opt_val, categorical_features)

    # PR-AUC 계산
    score = average_precision_score(y_opt_val, val_proba)

    return score


# 5. Optuna 최적화 실행
study = optuna.create_study(direction='maximize')
print('🔥 Optuna + 앙상블 하이퍼파라미터 탐색을 시작합니다...')
study.optimize(objective, n_trials=30)

print('\n🏆 [Best Trial]')
print(f'최고 PR-AUC 점수: {study.best_value:.4f}')
print('최적의 파라미터:')
for key, value in study.best_params.items():
    print(f'  {key}: {value}')

[I 2026-03-09 14:44:16,875] A new study created in memory with name: no-name-6ee826cd-18cc-4c4e-abcf-919e09e1f08c


🔥 Optuna + 앙상블 하이퍼파라미터 탐색을 시작합니다...


[W 2026-03-09 14:46:48,940] Trial 0 failed with parameters: {'iterations': 761, 'learning_rate': 0.11543528121749663, 'depth': 5, 'l2_leaf_reg': 13.232599495920065, 'random_strength': 0.016028360356297188, 'bagging_temperature': 0.6767872257562235} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "c:\potenup3\pj02_daangn-marke\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_64208\3054819905.py", line 61, in objective
    models = train_ensemble_for_optuna(X_opt_tr, y_opt_tr, params, n_models=5)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_64208\3054819905.py", line 27, in train_ensemble_for_optuna
    model.fit(train_pool, verbose=False)
  File "c:\potenup3\pj02_daangn-marke\.venv\Lib\site-packages\catboost\cor

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


# 1. Permutation Importance 계산 함수 정의
def get_ensemble_permutation_importance(
    models, X_val, y_val, cat_features, metric_fn=average_precision_score
):
    """
    각 피처를 무작위로 섞었을 때 평가지표(PR-AUC)가 얼마나 하락하는지 계산합니다.
    """
    # 1) 섞기 전 원본 데이터의 기준(Baseline) 점수 계산
    baseline_proba = predict_ensemble(models, X_val, cat_features)
    baseline_score = metric_fn(y_val, baseline_proba)

    importances = {}

    # 2) 각 컬럼을 순회하며 하나씩 섞어보고 점수 하락폭 측정
    for col in X_val.columns:
        X_val_shuffled = X_val.copy()

        # 해당 컬럼의 데이터만 무작위로 섞음 (.values를 써서 인덱스 꼬임 방지)
        X_val_shuffled[col] = np.random.permutation(X_val_shuffled[col].values)

        # 섞인 데이터로 예측 및 점수 계산
        shuffled_proba = predict_ensemble(models, X_val_shuffled, cat_features)
        shuffled_score = metric_fn(y_val, shuffled_proba)

        # 중요도 = 기존 점수 - 섞은 후 점수 (많이 떨어질수록 중요한 변수)
        importance_score = baseline_score - shuffled_score
        importances[col] = importance_score

    # 데이터프레임으로 변환 후 내림차순 정렬
    df_imp = (
        pd.DataFrame(
            {
                'Feature': list(importances.keys()),
                'Importance': list(importances.values()),
            }
        )
        .sort_values(by='Importance', ascending=False)
        .reset_index(drop=True)
    )

    return df_imp


# ==========================================
# 🚀 Optuna 완료 후 최종 모델 학습 및 중요도 시각화
# ==========================================

print('\n🌟 [최종 모델 학습 및 Permutation Importance 추출]')

# 1. Optuna에서 찾은 베스트 파라미터 가져오기
best_params = study.best_params
best_params['loss_function'] = 'Logloss'
best_params['eval_metric'] = 'Logloss'
best_params['task_type'] = 'GPU'
best_params['bootstrap_type'] = 'Bayesian'
best_params['verbose'] = False

# 2. 베스트 파라미터로 최종 앙상블 모델 학습
final_models = train_ensemble_for_optuna(
    X_opt_tr, y_opt_tr, best_params, categorical_features, n_models=5
)

# 3. 검증 데이터(X_opt_val)를 이용해 Permutation Importance 계산
perm_importances = get_ensemble_permutation_importance(
    final_models, X_opt_val, y_opt_val, categorical_features
)

# 4. 결과 시각화
plt.figure(figsize=(10, 8))
sns.barplot(
    x='Importance',
    y='Feature',
    data=perm_importances.head(20),  # 상위 20개만 출력
    palette='viridis',
)
plt.title('Permutation Importance (Ensemble Model, PR-AUC Drop)')
plt.xlabel('Decrease in PR-AUC Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

# 상세 수치 출력
print('\n[상세 변수 중요도 수치 (Top 10)]')
print(perm_importances.head(10))

<class 'pandas.DataFrame'>
Index: 131842 entries, 0 to 220687
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   title    131842 non-null  str  
 1   content  131842 non-null  str  
dtypes: str(2)
memory usage: 50.4 MB


In [ ]:
import os


def save_ensemble_models(models, save_dir='ensemble_models'):
    """
    학습된 앙상블 모델 리스트를 지정한 폴더에 저장합니다.
    """
    # 폴더가 없으면 생성
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    for i, model in enumerate(models):
        # catboost_model_0.cbm, catboost_model_1.cbm 형태로 저장
        file_path = os.path.join(save_dir, f'catboost_model_{i}.cbm')
        model.save_model(file_path)

    print(
        f"✅ {len(models)}개의 앙상블 모델이 '{save_dir}' 폴더에 안전하게 저장되었습니다."
    )


# 사용 예시 (위에서 학습한 final_models를 저장)
save_ensemble_models(final_models, save_dir='../data/models/my_best_ensemble')